In [7]:
import pandas as pd
import numpy as np


def clean_column_name(col):
    """Make Excel column names consistent."""

    col = str(col)

    # Remove invisible/problematic characters
    col = col.replace("\ufeff", "")
    col = col.replace("\xa0", " ")

    # Strip whitespace
    col = col.strip()

    # Lowercase
    col = col.lower()

    # Collapse repeated whitespace
    col = " ".join(col.split())

    return col


def load_excel(path, experiment_name):

    df = pd.read_excel(path)

    # --------------------------------------------------
    # Clean column names
    # --------------------------------------------------

    df.columns = [
        clean_column_name(c)
        for c in df.columns
    ]

    # Remove empty rows and columns
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")

    print(f"\n{experiment_name}")
    print("Columns found:")
    print(df.columns.tolist())

    # ==================================================
    # FORMAT 1
    # Fixed threshold summary
    #
    # micro_f1_all
    # macro_f1_all
    # weighted_f1_all
    # ==================================================

    if "micro_f1_all" in df.columns:

        # The first column contains dataset names,
        # although its actual Excel name contains
        # threshold information.
        first_col = df.columns[0]

        df = df.rename(
            columns={
                first_col: "dataset"
            }
        )

        rows = []

        for _, r in df.iterrows():

            rows.append({

                "experiment": experiment_name,
                "experiment_group": "fixed_threshold",

                "dataset": r["dataset"],

                "method": "fixed",
                "aggregation": None,
                "threshold": None,

                # ALL metrics
                "macro_f1": r["macro_f1_all"],
                "micro_f1": r["micro_f1_all"],
                "weighted_f1": r["weighted_f1_all"],

                # MAIN metrics
                "macro_f1_main": r["macro_f1_main"],
                "micro_f1_main": r["micro_f1_main"],
                "weighted_f1_main": r["weighted_f1_main"],

                # Not available in this format
                "macro_precision": np.nan,
                "macro_recall": np.nan,
                "micro_precision": np.nan,
                "micro_recall": np.nan,
            })

        return pd.DataFrame(rows)


    # ==================================================
    # FORMAT 3
    #
    # dataset
    # aggregation
    # threshold
    # threshold_method
    #
    # This MUST be checked before Format 2 because
    # there is no "method" column.
    # ==================================================

    elif (
        "aggregation" in df.columns
        and "threshold_method" in df.columns
    ):

        df["experiment"] = experiment_name
        df["experiment_group"] = "optimized_global"

        # Use threshold_method as our normalized "method"
        df["method"] = df["threshold_method"]

        # These files do not have MAIN metrics
        df["macro_f1_main"] = np.nan
        df["micro_f1_main"] = np.nan
        df["weighted_f1_main"] = np.nan

        return df[
            [
                "experiment",
                "experiment_group",
                "dataset",

                "method",
                "aggregation",
                "threshold",

                "macro_f1",
                "micro_f1",
                "weighted_f1",

                "macro_precision",
                "macro_recall",
                "micro_precision",
                "micro_recall",

                "macro_f1_main",
                "micro_f1_main",
                "weighted_f1_main",
            ]
        ]


    # ==================================================
    # FORMAT 2
    #
    # dataset
    # method
    # threshold
    #
    # constant / global / per_class
    # ==================================================

    elif "method" in df.columns:

        # Make sure dataset exists
        if "dataset" not in df.columns:

            first_col = df.columns[0]

            print(
                f"WARNING: 'dataset' not found. "
                f"Using first column '{first_col}' as dataset."
            )

            df = df.rename(
                columns={
                    first_col: "dataset"
                }
            )

        df["experiment"] = experiment_name
        df["experiment_group"] = "threshold_strategy"

        # No aggregation in this format
        df["aggregation"] = None

        # No MAIN metrics in this format
        df["macro_f1_main"] = np.nan
        df["micro_f1_main"] = np.nan
        df["weighted_f1_main"] = np.nan

        return df[
            [
                "experiment",
                "experiment_group",
                "dataset",

                "method",
                "aggregation",
                "threshold",

                "macro_f1",
                "micro_f1",
                "weighted_f1",

                "macro_precision",
                "macro_recall",
                "micro_precision",
                "micro_recall",

                "macro_f1_main",
                "micro_f1_main",
                "weighted_f1_main",
            ]
        ]


    # ==================================================
    # Unknown format
    # ==================================================

    raise ValueError(
        f"\nCould not recognize format of:\n{path}\n\n"
        f"Columns found:\n{df.columns.tolist()}"
    )


In [13]:
files = {
    "Experiment 1": r"C:\Users\alrazz\Downloads\Results\Modified Files\General results\MultiCore_XLMR_base_evaluation.xlsx",
    "Experiment 2": r"C:\Users\alrazz\Downloads\Results\Modified Files\General results\MultiCore_BGEM3_base_evaluation.xlsx",
    "Experiment 3": r"C:\Users\alrazz\Downloads\Results\Modified Files\General results\MultiCore_BGEM3_base_evaluation_n512_TH035.xlsx",
    "Experiment 4": r"C:\Users\alrazz\Downloads\Results\Modified Files\General results\MultiCore_XLMR_Finetuned_Threshold_tuned.xlsx",
    "Experiment 5": r"C:\Users\alrazz\Downloads\Results\Modified Files\General results\MultiCore_BGEM3_Finetuned_Threshold_tuned.xlsx",
    "Experiment 6": r"C:\Users\alrazz\Downloads\Results\Modified Files\General results\MultiCore_BGEM3_Sliding_Finetuned_Threshold_tuned.xlsx",
}

dfs = [
    load_excel(path, name)
    for name, path in files.items()
]

data = pd.concat(dfs, ignore_index=True)



Experiment 1
Columns found:
['dataset (0.35 =threshold, max_length=512)', 'micro_f1_all', 'macro_f1_all', 'weighted_f1_all', 'micro_f1_main', 'macro_f1_main', 'weighted_f1_main']

Experiment 2
Columns found:
['dataset (0.4 =threshold, max_length=1024))', 'micro_f1_all', 'macro_f1_all', 'weighted_f1_all', 'micro_f1_main', 'macro_f1_main', 'weighted_f1_main']

Experiment 3
Columns found:
['dataset (0.35=threshold, max_length=512)', 'micro_f1_all', 'macro_f1_all', 'weighted_f1_all', 'micro_f1_main', 'macro_f1_main', 'weighted_f1_main']

Experiment 4
Columns found:
['dataset', 'method', 'threshold', 'macro_f1', 'micro_f1', 'weighted_f1', 'macro_precision', 'macro_recall', 'micro_precision', 'micro_recall']

Experiment 5
Columns found:
['dataset', 'method', 'threshold', 'macro_f1', 'micro_f1', 'weighted_f1', 'macro_precision', 'macro_recall', 'micro_precision', 'micro_recall']

Experiment 6
Columns found:
['dataset', 'aggregation', 'threshold', 'threshold_method', 'macro_f1', 'micro_f1', '

In [10]:
import plotly.io as pio

pio.renderers.default = "browser"


In [20]:
experiment_labels = {
    "Experiment 1": "XLM-R Base (512, 0.4)",
    "Experiment 2": "BGE-M3 Base (1024, 0.4)",
    "Experiment 3": "BGE-M3 Base (512, 0.35)",
    "Experiment 4": "XLM-R FT + Thresh",
    "Experiment 5": "BGE-M3 FT + Thresh",
    "Experiment 6": "BGE-M3 Sliding FT + Thresh",
}


In [21]:
data["experiment_label"] = data["experiment"].map(experiment_labels)


In [22]:
group_labels = {
    "fixed_threshold": "No Fine-tuning",
    "threshold_strategy": "Fine-tuned + Threshold Tuning",
    "optimized_global": "Fine-tuned + Threshold Tuning + Sliding Window",
}

data["experiment_group_label"] = data["experiment_group"].map(group_labels)


# Macro_F1 plots

In [ ]:
import plotly.express as px

overview = (
    data
    .sort_values("macro_f1", ascending=False)
    .groupby(
        [
            "experiment",
            "experiment_label",
            "experiment_group",
            "experiment_group_label",
            "dataset"
        ],
        as_index=False
    )
    .first()
)


fig = px.bar(
    overview,

    x="dataset",
    y="macro_f1",

    color="experiment_label",

    facet_col="experiment_group_label",

    barmode="group",

    hover_data={
        "experiment_label": True,
        "experiment": False,
        "experiment_group": False,
        "experiment_group_label": False,

        "dataset": True,

        "macro_f1": ":.4f",
        "micro_f1": ":.4f",
        "weighted_f1": ":.4f",

        "method": True,
        "aggregation": True,
        "threshold": True,
    },

    labels={
        "macro_f1": "Macro F1",
        "dataset": "Dataset",
        "experiment_label": "Experiment",
        "experiment_group_label": "",
        "method": "Threshold Method",
        "aggregation": "Aggregation",
        "threshold": "Threshold",
    },

    title="Best Macro F1 Across Experimental Setups"
)


fig.update_yaxes(range=[0, 1])


fig.update_layout(
    height=650,
    hovermode="closest",
)


fig.show()


# Macro and Micro F1 plots (all labels)

## Seprate

In [26]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [59]:
def plot_metric(overview, metric, title):

    fig = px.bar(
        overview,

        x="dataset",
        y=metric,

        color="experiment_label",

        facet_col="experiment_group_label",

        barmode="group",

        hover_data={
            "experiment_label": True,
            "experiment": False,
            "experiment_group": False,
            "experiment_group_label": False,

            "dataset": True,

            "macro_f1": ":.4f",
            "micro_f1": ":.4f",
            "weighted_f1": ":.4f",

            "method": True,
            "aggregation": True,
            "threshold": True,
        },

        labels={
            metric: metric.replace("_", " ").title(),
            "dataset": "Dataset",
            "experiment_label": "Experiment",
            "experiment_group_label": "",
        },

        title=title
    )

    fig.update_yaxes(range=[0, 1])

    fig.update_layout(
        height=650,
        hovermode="closest",
    )

    fig.write_html(
    fr"C:\Users\alrazz\Downloads\Results\Modified Files\General results\{metric}.html",
    include_plotlyjs=True
)


    fig.show()


In [ ]:
plot_metric(
    overview,
    "macro_f1",
    "Best Macro F1 Across Experimental Setups"
)

plot_metric(
    overview,
    "micro_f1",
    "Best Micro F1 Across Experimental Setups"
)


## Aggregation add

In [71]:
overview = (
    data
    .sort_values("macro_f1", ascending=False)
    .groupby(
        [
            "experiment",
            "experiment_label",
            "experiment_group",
            "experiment_group_label",
            "dataset",
            "aggregation"
        ],
        as_index=False,
        dropna=False
    )
    .first()
)


In [72]:
overview["plot_label"] = overview["experiment_label"]

mask = overview["experiment"] == "Experiment 6"

overview.loc[mask, "plot_label"] = (
    overview.loc[mask, "experiment_label"]
    + " — "
    + overview.loc[mask, "aggregation"]
)


In [73]:
print(
    overview[
        overview["experiment"] == "Experiment 6"
    ][
        ["dataset", "aggregation", "macro_f1", "micro_f1", "plot_label"]
    ].to_string(index=False)
)


     dataset aggregation  macro_f1  micro_f1                             plot_label
   ID_hybrid         max  0.724113  0.737335       BGE-M3 Sliding FT + Thresh — max
   ID_hybrid        mean  0.749698  0.760401      BGE-M3 Sliding FT + Thresh — mean
   ID_hybrid   top2_mean  0.730118  0.741674 BGE-M3 Sliding FT + Thresh — top2_mean
SP_ID_hybrid         max  0.721300  0.733776       BGE-M3 Sliding FT + Thresh — max
SP_ID_hybrid        mean  0.743721  0.754813      BGE-M3 Sliding FT + Thresh — mean
SP_ID_hybrid   top2_mean  0.724426  0.737006 BGE-M3 Sliding FT + Thresh — top2_mean
   SP_hybrid         max  0.703077  0.715729       BGE-M3 Sliding FT + Thresh — max
   SP_hybrid        mean  0.725013  0.739155      BGE-M3 Sliding FT + Thresh — mean
   SP_hybrid   top2_mean  0.704008  0.719705 BGE-M3 Sliding FT + Thresh — top2_mean
   no_hybrid         max  0.721855  0.734074       BGE-M3 Sliding FT + Thresh — max
   no_hybrid        mean  0.748744  0.761781      BGE-M3 Sliding FT + Thresh

In [74]:
def plot_metric(overview, metric, title):

    fig = px.bar(
        overview,

        x="dataset",
        y=metric,

        color="plot_label",

        facet_col="experiment_group_label",

        barmode="group",

        hover_data={
            "plot_label": True,
            "experiment_label": True,

            "experiment": False,
            "experiment_group": False,
            "experiment_group_label": False,

            "dataset": True,

            "macro_f1": ":.4f",
            "micro_f1": ":.4f",
            "weighted_f1": ":.4f",

            "method": True,
            "aggregation": True,
            "threshold": True,
        },

        labels={
            metric: metric.replace("_", " ").title(),
            "dataset": "Dataset",
            "plot_label": "Experiment",
            "experiment_label": "Experiment",
            "experiment_group_label": "",
            "method": "Threshold Method",
            "aggregation": "Aggregation",
            "threshold": "Threshold",
        },

        title=title
    )

    fig.update_yaxes(range=[0, 1])

    fig.update_layout(
        height=650,
        hovermode="closest",
    )

    fig.write_html(
        fr"C:\Users\alrazz\Downloads\Results\Modified Files\General results\{metric}_with_agg.html",
        include_plotlyjs=True
    )

    fig.show()


In [75]:
plot_metric(
    overview,
    "macro_f1",
    "Macro F1 Across Experimental Setups"
)

plot_metric(
    overview,
    "micro_f1",
    "Micro F1 Across Experimental Setups"
)


## In one page

In [56]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_f1_overview(overview, selected_experiments=None):

    # --------------------------------------------------
    # Filter experiments
    # --------------------------------------------------

    if selected_experiments is not None:
        overview = overview[
            overview["experiment_label"].isin(selected_experiments)
        ].copy()

    # --------------------------------------------------
    # Experimental groups
    # --------------------------------------------------

    groups = (
        overview["experiment_group_label"]
        .dropna()
        .unique()
        .tolist()
    )

    # --------------------------------------------------
    # Create subplot figure
    # --------------------------------------------------

    fig = make_subplots(
        rows=2,
        cols=len(groups),

        shared_yaxes=True,

        horizontal_spacing=0.04,
        vertical_spacing=0.10,

        subplot_titles=groups
    )

    # --------------------------------------------------
    # Experiment colors
    # --------------------------------------------------

    experiments = (
        overview["experiment_label"]
        .dropna()
        .unique()
        .tolist()
    )

    colors = px.colors.qualitative.Plotly

    color_map = {
        exp: colors[i % len(colors)]
        for i, exp in enumerate(experiments)
    }

    # --------------------------------------------------
    # Add bars
    # --------------------------------------------------

    for col, group in enumerate(groups, start=1):

        group_data = overview[
            overview["experiment_group_label"] == group
        ]

        for experiment in experiments:

            subset = group_data[
                group_data["experiment_label"] == experiment
            ]

            if subset.empty:
                continue

            customdata = subset[
                [
                    "experiment_label",
                    "dataset",
                    "macro_f1",
                    "micro_f1",
                    "weighted_f1",
                    "method",
                    "aggregation",
                    "threshold"
                ]
            ].values

            # ==========================================
            # Macro F1
            # ==========================================

            fig.add_trace(
                go.Bar(
                    x=subset["dataset"],
                    y=subset["macro_f1"],

                    name=experiment,
                    legendgroup=experiment,

                    showlegend=True,

                    marker_color=color_map[experiment],

                    customdata=customdata,

                    hovertemplate=(
                        "<b>%{customdata[0]}</b><br>"
                        "Dataset: %{customdata[1]}<br>"
                        "<br>"
                        "<b>Macro F1:</b> %{customdata[2]:.4f}<br>"
                        "Micro F1: %{customdata[3]:.4f}<br>"
                        "Weighted F1: %{customdata[4]:.4f}<br>"
                        "<br>"
                        "Threshold method: %{customdata[5]}<br>"
                        "Aggregation: %{customdata[6]}<br>"
                        "Threshold: %{customdata[7]}<br>"
                        "<extra></extra>"
                    )
                ),
                row=1,
                col=col
            )

            # ==========================================
            # Micro F1
            # ==========================================

            fig.add_trace(
                go.Bar(
                    x=subset["dataset"],
                    y=subset["micro_f1"],

                    name=experiment,
                    legendgroup=experiment,

                    showlegend=False,

                    marker_color=color_map[experiment],

                    customdata=customdata,

                    hovertemplate=(
                        "<b>%{customdata[0]}</b><br>"
                        "Dataset: %{customdata[1]}<br>"
                        "<br>"
                        "<b>Micro F1:</b> %{customdata[3]:.4f}<br>"
                        "Macro F1: %{customdata[2]:.4f}<br>"
                        "Weighted F1: %{customdata[4]:.4f}<br>"
                        "<br>"
                        "Threshold method: %{customdata[5]}<br>"
                        "Aggregation: %{customdata[6]}<br>"
                        "Threshold: %{customdata[7]}<br>"
                        "<extra></extra>"
                    )
                ),
                row=2,
                col=col
            )

    # --------------------------------------------------
    # Y axes
    # --------------------------------------------------

    fig.update_yaxes(
        range=[0, 1],
        title_text="F1 Macro",
        row=1,
        col=1
    )

    fig.update_yaxes(
        range=[0, 1],
        title_text="F1 Micro",
        row=2,
        col=1
    )

    # --------------------------------------------------
    # X axes
    # --------------------------------------------------

    for col in range(1, len(groups) + 1):

        fig.update_xaxes(
            title_text="Dataset",
            row=2,
            col=col
        )

    # --------------------------------------------------
    # Layout
    # --------------------------------------------------

    fig.update_layout(

        title=dict(
            text="Best F1 Performance Across Multiple Experiments",
            x=0.5,
            xanchor="center"
        ),

        height=850,

        barmode="group",

        hovermode="closest",

        legend=dict(
            title=dict(
                text="<b>Experiment</b>"
            ),

            orientation="v",

            yanchor="top",
            y=1,

            xanchor="left",
            x=1.02,

            bgcolor="rgba(255,255,255,0.8)",

            bordercolor="lightgray",
            borderwidth=1
        ),

        margin=dict(
            t=120,
            b=90,
            l=80,
            r=300
        )
    )

    # --------------------------------------------------
    # Row labels
    # --------------------------------------------------

    fig.add_annotation(
        x=-0.065,
        y=0.75,
        xref="paper",
        yref="paper",

        text="<b>Macro F1</b>",

        textangle=-90,

        showarrow=False
    )

    fig.add_annotation(
        x=-0.065,
        y=0.25,
        xref="paper",
        yref="paper",

        text="<b>Micro F1</b>",

        textangle=-90,

        showarrow=False
    )

    fig.write_html(
    r"C:\Users\alrazz\Downloads\Results\Modified Files\General results\F1_overview.html",
    include_plotlyjs=True
)


    fig.show()


In [57]:
plot_f1_overview(overview)


## With agg

In [78]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_f1_overview(overview, selected_experiments=None):

    # --------------------------------------------------
    # Filter experiments
    # --------------------------------------------------

    if selected_experiments is not None:
        overview = overview[
            overview["experiment_label"].isin(selected_experiments)
        ].copy()

    # --------------------------------------------------
    # Experimental groups
    # --------------------------------------------------

    groups = (
        overview["experiment_group_label"]
        .dropna()
        .unique()
        .tolist()
    )

    # --------------------------------------------------
    # Create subplot figure
    # --------------------------------------------------

    fig = make_subplots(
        rows=2,
        cols=len(groups),

        shared_yaxes=True,

        horizontal_spacing=0.04,
        vertical_spacing=0.10,

        subplot_titles=groups
    )

    # --------------------------------------------------
    # Create plotting labels
    #
    # Normal experiments:
    #     XLM-R Base (512, 0.4)
    #
    # Experiment 6:
    #     BGE-M3 Sliding FT + Thresh — max
    #     BGE-M3 Sliding FT + Thresh — mean
    #     BGE-M3 Sliding FT + Thresh — top2_mean
    # --------------------------------------------------

    overview = overview.copy()

    overview["plot_label"] = overview["experiment_label"]

    mask = overview["experiment"] == "Experiment 6"

    overview.loc[mask, "plot_label"] = (
        overview.loc[mask, "experiment_label"]
        + " — "
        + overview.loc[mask, "aggregation"]
    )

    # --------------------------------------------------
    # Experiments / plotting labels
    # --------------------------------------------------

    plot_labels = (
        overview["plot_label"]
        .dropna()
        .unique()
        .tolist()
    )

    # --------------------------------------------------
    # Experiment colors
    # --------------------------------------------------

    colors = px.colors.qualitative.Plotly

    # Give each original experiment a base color
    experiments = (
        overview["experiment_label"]
        .dropna()
        .unique()
        .tolist()
    )

    color_map = {
        label: colors[i % len(colors)]
        for i, label in enumerate(plot_labels)
    }

    # --------------------------------------------------
    # Add bars
    # --------------------------------------------------

    for col, group in enumerate(groups, start=1):

        group_data = overview[
            overview["experiment_group_label"] == group
        ]

        for plot_label in plot_labels:

            subset = group_data[
                group_data["plot_label"] == plot_label
            ]

            if subset.empty:
                continue

            # ------------------------------------------
            # Determine original experiment
            # ------------------------------------------

            experiment = subset["experiment_label"].iloc[0]

            # ------------------------------------------
            # Custom hover data
            # ------------------------------------------

            customdata = subset[
                [
                    "experiment_label",
                    "dataset",
                    "macro_f1",
                    "micro_f1",
                    "weighted_f1",
                    "method",
                    "aggregation",
                    "threshold"
                ]
            ].values

            # ==========================================
            # Macro F1
            # ==========================================

            fig.add_trace(
                go.Bar(
                    x=subset["dataset"],
                    y=subset["macro_f1"],

                    name=plot_label,
                    legendgroup=plot_label,

                    showlegend=True,

                    marker_color=color_map[plot_label],

                    customdata=customdata,

                    hovertemplate=(
                        "<b>%{customdata[0]}</b><br>"
                        "Dataset: %{customdata[1]}<br>"
                        "<br>"
                        "<b>Macro F1:</b> %{customdata[2]:.4f}<br>"
                        "Micro F1: %{customdata[3]:.4f}<br>"
                        "Weighted F1: %{customdata[4]:.4f}<br>"
                        "<br>"
                        "Threshold method: %{customdata[5]}<br>"
                        "Aggregation: %{customdata[6]}<br>"
                        "Threshold: %{customdata[7]}<br>"
                        "<extra></extra>"
                    )
                ),
                row=1,
                col=col
            )

            # ==========================================
            # Micro F1
            # ==========================================

            fig.add_trace(
                go.Bar(
                    x=subset["dataset"],
                    y=subset["micro_f1"],

                    name=plot_label,
                    legendgroup=plot_label,

                    showlegend=False,

                    marker_color=color_map[plot_label],

                    customdata=customdata,

                    hovertemplate=(
                        "<b>%{customdata[0]}</b><br>"
                        "Dataset: %{customdata[1]}<br>"
                        "<br>"
                        "<b>Micro F1:</b> %{customdata[3]:.4f}<br>"
                        "Macro F1: %{customdata[2]:.4f}<br>"
                        "Weighted F1: %{customdata[4]:.4f}<br>"
                        "<br>"
                        "Threshold method: %{customdata[5]}<br>"
                        "Aggregation: %{customdata[6]}<br>"
                        "Threshold: %{customdata[7]}<br>"
                        "<extra></extra>"
                    )
                ),
                row=2,
                col=col
            )

    # --------------------------------------------------
    # Y axes
    # --------------------------------------------------

    fig.update_yaxes(
        range=[0, 1],
        title_text="F1 Macro",
        row=1,
        col=1
    )

    fig.update_yaxes(
        range=[0, 1],
        title_text="F1 Micro",
        row=2,
        col=1
    )

    # --------------------------------------------------
    # X axes
    # --------------------------------------------------

    for col in range(1, len(groups) + 1):

        fig.update_xaxes(
            title_text="Dataset",
            row=2,
            col=col
        )

    # --------------------------------------------------
    # Layout
    # --------------------------------------------------

    fig.update_layout(

        title=dict(
            text="Best F1 Performance Across Multiple Experiments",
            x=0.5,
            xanchor="center"
        ),

        height=850,

        barmode="group",

        hovermode="closest",

        legend=dict(
            title=dict(
                text="<b>Experiment</b>"
            ),

            orientation="v",

            yanchor="top",
            y=1,

            xanchor="left",
            x=1.02,

            bgcolor="rgba(255,255,255,0.8)",

            bordercolor="lightgray",
            borderwidth=1
        ),

        margin=dict(
            t=120,
            b=90,
            l=80,
            r=300
        )
    )

    # --------------------------------------------------
    # Row labels
    # --------------------------------------------------

    fig.add_annotation(
        x=-0.065,
        y=0.75,
        xref="paper",
        yref="paper",

        text="<b>Macro F1</b>",

        textangle=-90,

        showarrow=False
    )

    fig.add_annotation(
        x=-0.065,
        y=0.25,
        xref="paper",
        yref="paper",

        text="<b>Micro F1</b>",

        textangle=-90,

        showarrow=False
    )

    # --------------------------------------------------
    # Save
    # --------------------------------------------------

    fig.write_html(
        r"C:\Users\alrazz\Downloads\Results\Modified Files\General results\F1_overview_with_agg.html",
        include_plotlyjs=True
    )

    fig.show()


In [79]:
plot_f1_overview(overview)

# Perclass

## _MultiCore_BGE_m3_base_evaluation

In [81]:
from pathlib import Path

DATA_DIR = Path(r"C:\Users\alrazz\Downloads\Results\_MultiCore_BGE_m3_base_evaluation")

print("Files in folder:")
for file in DATA_DIR.iterdir():
    print(file.name)


Files in folder:
Combined_hybrid_no_NA_classification_report.csv
Combined_hybrid_no_NA_classification_report.json
Combined_hybrid_no_NA_metrics.json
Combined_hybrid_no_NA_predictions.csv
Combined_ID_hybrid_no_NA_classification_report.csv
Combined_ID_hybrid_no_NA_classification_report.json
Combined_ID_hybrid_no_NA_metrics.json
Combined_ID_hybrid_no_NA_predictions.csv
Combined_single_no_NA_classification_report.csv
Combined_single_no_NA_classification_report.json
Combined_single_no_NA_metrics.json
Combined_single_no_NA_predictions.csv
Combined_SP_hybrid_no_NA_classification_report.csv
Combined_SP_hybrid_no_NA_classification_report.json
Combined_SP_hybrid_no_NA_metrics.json
Combined_SP_hybrid_no_NA_predictions.csv
model_comparison.csv
model_comparison.json


In [82]:
DATA_DIR = Path(
    r"C:\Users\alrazz\Downloads\Results\_MultiCore_BGE_m3_base_evaluation"
)

files = {
    "Hybrid": DATA_DIR / "Combined_hybrid_no_NA_classification_report.csv",
    "ID Hybrid": DATA_DIR / "Combined_ID_hybrid_no_NA_classification_report.csv",
    "Single": DATA_DIR / "Combined_single_no_NA_classification_report.csv",
    "SP Hybrid": DATA_DIR / "Combined_SP_hybrid_no_NA_classification_report.csv",
}

In [85]:
def load_classification_report(filepath):

    df = pd.read_csv(
    filepath,
    index_col=0,
    keep_default_na=False
)


    # Remove summary rows
    summary_rows = [
        "micro avg",
        "macro avg",
        "weighted avg",
        "samples avg"
    ]

    df = df[~df.index.isin(summary_rows)].copy()

    return df


reports = {}

for name, filepath in files.items():

    if not filepath.exists():
        raise FileNotFoundError(f"File not found: {filepath}")

    reports[name] = load_classification_report(filepath)

    print(f"Loaded: {name}")
    print(reports[name].shape)


# =========================================================
# 3. Combine everything into one DataFrame
# =========================================================

records = []

for variation, df in reports.items():

    for label, row in df.iterrows():

        records.append({
            "variation": variation,
            "label": label,
            "precision": row["precision"],
            "recall": row["recall"],
            "f1-score": row["f1-score"],
            "support": row["support"],
        })


combined = pd.DataFrame(records)

print("\nCombined data:")
display(combined.head())

Loaded: Hybrid
(25, 4)
Loaded: ID Hybrid
(25, 4)
Loaded: Single
(25, 4)
Loaded: SP Hybrid
(25, 4)

Combined data:


,variation,label,precision,recall,f1-score,support
0,Hybrid,MT,0.857143,0.193548,0.315789,31.0
1,Hybrid,LY,1.000000,0.450980,0.621622,51.0
2,Hybrid,SP,0.909091,0.491803,0.638298,61.0
3,Hybrid,ID,0.357143,0.172414,0.232558,58.0
4,Hybrid,NA,0.889764,0.582474,0.704050,194.0


In [86]:
# =========================================================
# 4. Create interactive Plotly figure
# =========================================================

metrics = ["precision", "recall", "f1-score"]

colors = {
    "Hybrid": "#636EFA",
    "ID Hybrid": "#EF553B",
    "Single": "#00CC96",
    "SP Hybrid": "#AB63FA",
}

fig = go.Figure()

# Add traces
for metric in metrics:

    for variation in files.keys():

        df = combined[
            combined["variation"] == variation
        ]

        fig.add_trace(
            go.Bar(
                x=df["label"],
                y=df[metric],
                name=variation,
                marker_color=colors[variation],

                # Put values above bars
                text=df[metric].round(2),
                textposition="outside",

                hovertemplate=(
                    "<b>Label:</b> %{x}<br>"
                    f"<b>{variation}</b><br>"
                    f"<b>{metric}:</b> %{{y:.3f}}"
                    "<extra></extra>"
                ),

                visible=(metric == "f1-score")
            )
        )


# =========================================================
# 5. Dropdown menu
# =========================================================

buttons = []

n_variations = len(files)

for metric_index, metric in enumerate(metrics):

    visibility = [False] * (len(metrics) * n_variations)

    start = metric_index * n_variations
    end = start + n_variations

    visibility[start:end] = [True] * n_variations

    buttons.append(
        dict(
            label=metric.title(),
            method="update",

            args=[
                {"visible": visibility},

                {
                    "title": (
                        f"{metric.title()} Comparison "
                        "Across Models"
                    ),

                    "yaxis": {
                        "title": metric.title(),
                        "range": [0, 1.1]
                    }
                }
            ]
        )
    )


# =========================================================
# 6. Layout
# =========================================================

fig.update_layout(

    title="F1-score Comparison Across Models",

    xaxis=dict(
        title="Label",
        tickangle=-45
    ),

    yaxis=dict(
        title="F1-score",
        range=[0, 1.1]
    ),

    barmode="group",

    template="plotly_white",

    height=750,

    legend=dict(
        title="Model",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0
    ),

    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1,
            xanchor="right",
            y=1.12,
            yanchor="top"
        )
    ],

    margin=dict(
        l=70,
        r=40,
        t=120,
        b=120
    )
)

fig.show()


## _MultiCore_XLMR_base_evaluation

In [90]:
DATA_DIR = Path(
    r"C:\Users\alrazz\Downloads\Results\_MultiCore_XLMR_base_evaluation"
)

files = {
    "Hybrid": DATA_DIR / "Combined_hybrid_no_NA_classification_report.csv",
    "ID Hybrid": DATA_DIR / "Combined_ID_hybrid_no_NA_classification_report.csv",
    "Single": DATA_DIR / "Combined_single_no_NA_classification_report.csv",
    "SP Hybrid": DATA_DIR / "Combined_SP_hybrid_no_NA_classification_report.csv",
}

In [91]:
reports = {}

for name, filepath in files.items():

    if not filepath.exists():
        raise FileNotFoundError(f"File not found: {filepath}")

    reports[name] = load_classification_report(filepath)

    print(f"Loaded: {name}")
    print(reports[name].shape)


# =========================================================
# 3. Combine everything into one DataFrame
# =========================================================

records = []

for variation, df in reports.items():

    for label, row in df.iterrows():

        records.append({
            "variation": variation,
            "label": label,
            "precision": row["precision"],
            "recall": row["recall"],
            "f1-score": row["f1-score"],
            "support": row["support"],
        })


combined = pd.DataFrame(records)

print("\nCombined data:")
display(combined.head())

Loaded: Hybrid
(25, 4)
Loaded: ID Hybrid
(25, 4)
Loaded: Single
(25, 4)
Loaded: SP Hybrid
(25, 4)

Combined data:


,variation,label,precision,recall,f1-score,support
0,Hybrid,MT,0.818182,0.290323,0.428571,31.0
1,Hybrid,LY,0.862069,0.490196,0.625000,51.0
2,Hybrid,SP,0.893617,0.688525,0.777778,61.0
3,Hybrid,ID,0.222222,0.137931,0.170213,58.0
4,Hybrid,NA,0.835443,0.680412,0.750000,194.0


In [92]:
# =========================================================
# 4. Create interactive Plotly figure
# =========================================================

metrics = ["precision", "recall", "f1-score"]

colors = {
    "Hybrid": "#636EFA",
    "ID Hybrid": "#EF553B",
    "Single": "#00CC96",
    "SP Hybrid": "#AB63FA",
}

fig = go.Figure()

# Add traces
for metric in metrics:

    for variation in files.keys():

        df = combined[
            combined["variation"] == variation
        ]

        fig.add_trace(
            go.Bar(
                x=df["label"],
                y=df[metric],
                name=variation,
                marker_color=colors[variation],

                # Put values above bars
                text=df[metric].round(2),
                textposition="outside",

                hovertemplate=(
                    "<b>Label:</b> %{x}<br>"
                    f"<b>{variation}</b><br>"
                    f"<b>{metric}:</b> %{{y:.3f}}"
                    "<extra></extra>"
                ),

                visible=(metric == "f1-score")
            )
        )


# =========================================================
# 5. Dropdown menu
# =========================================================

buttons = []

n_variations = len(files)

for metric_index, metric in enumerate(metrics):

    visibility = [False] * (len(metrics) * n_variations)

    start = metric_index * n_variations
    end = start + n_variations

    visibility[start:end] = [True] * n_variations

    buttons.append(
        dict(
            label=metric.title(),
            method="update",

            args=[
                {"visible": visibility},

                {
                    "title": (
                        f"{metric.title()} Comparison "
                        "Across Models"
                    ),

                    "yaxis": {
                        "title": metric.title(),
                        "range": [0, 1.1]
                    }
                }
            ]
        )
    )


# =========================================================
# 6. Layout
# =========================================================

fig.update_layout(

    title="F1-score Comparison Across Models",

    xaxis=dict(
        title="Label",
        tickangle=-45
    ),

    yaxis=dict(
        title="F1-score",
        range=[0, 1.1]
    ),

    barmode="group",

    template="plotly_white",

    height=750,

    legend=dict(
        title="Model",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0
    ),

    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=1,
            xanchor="right",
            y=1.12,
            yanchor="top"
        )
    ],

    margin=dict(
        l=70,
        r=40,
        t=120,
        b=120
    )
)

fig.show()


In [ ]:
##proper way

In [93]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path


def load_classification_report(filepath):

    df = pd.read_csv(
        filepath,
        index_col=0,
        keep_default_na=False   # Important: keeps "NA" as a class
    )

    # Remove summary rows
    summary_rows = [
        "micro avg",
        "macro avg",
        "weighted avg",
        "samples avg"
    ]

    df = df[~df.index.isin(summary_rows)].copy()

    return df


In [94]:
def load_experiment(DATA_DIR):

    files = {
        "Hybrid": DATA_DIR / "Combined_hybrid_no_NA_classification_report.csv",
        "ID Hybrid": DATA_DIR / "Combined_ID_hybrid_no_NA_classification_report.csv",
        "Single": DATA_DIR / "Combined_single_no_NA_classification_report.csv",
        "SP Hybrid": DATA_DIR / "Combined_SP_hybrid_no_NA_classification_report.csv",
    }

    reports = {}

    for name, filepath in files.items():

        if not filepath.exists():
            raise FileNotFoundError(
                f"File not found: {filepath}"
            )

        reports[name] = load_classification_report(filepath)

    # Combine everything
    records = []

    for variation, df in reports.items():

        for label, row in df.iterrows():

            records.append({
                "variation": variation,
                "label": label,
                "precision": row["precision"],
                "recall": row["recall"],
                "f1-score": row["f1-score"],
                "support": row["support"],
            })

    combined = pd.DataFrame(records)

    return combined


In [95]:
def create_classification_plot(combined, title):

    metrics = ["precision", "recall", "f1-score"]

    colors = {
        "Hybrid": "#636EFA",
        "ID Hybrid": "#EF553B",
        "Single": "#00CC96",
        "SP Hybrid": "#AB63FA",
    }

    variations = list(colors.keys())

    fig = go.Figure()

    # -----------------------------------------------------
    # Add traces
    # -----------------------------------------------------

    for metric in metrics:

        for variation in variations:

            df = combined[
                combined["variation"] == variation
            ]

            fig.add_trace(
                go.Bar(
                    x=df["label"],
                    y=df[metric],
                    name=variation,
                    marker_color=colors[variation],

                    text=df[metric].round(2),
                    textposition="outside",

                    hovertemplate=(
                        "<b>Label:</b> %{x}<br>"
                        f"<b>{variation}</b><br>"
                        f"<b>{metric}:</b> %{{y:.3f}}"
                        "<extra></extra>"
                    ),

                    visible=(metric == "f1-score")
                )
            )

    # -----------------------------------------------------
    # Dropdown
    # -----------------------------------------------------

    buttons = []

    n_variations = len(variations)

    for metric_index, metric in enumerate(metrics):

        visibility = [
            False
        ] * (len(metrics) * n_variations)

        start = metric_index * n_variations
        end = start + n_variations

        visibility[start:end] = [
            True
        ] * n_variations

        buttons.append(
            dict(
                label=metric.title(),
                method="update",

                args=[
                    {"visible": visibility},

                    {
                        "title": (
                            f"{metric.title()} "
                            f"Comparison — {title}"
                        ),

                        "yaxis": {
                            "title": metric.title(),
                            "range": [0, 1.1]
                        }
                    }
                ]
            )
        )

    # -----------------------------------------------------
    # Layout
    # -----------------------------------------------------

    fig.update_layout(

        title=f"F1-score Comparison — {title}",

        xaxis=dict(
            title="Label",
            tickangle=-45
        ),

        yaxis=dict(
            title="F1-score",
            range=[0, 1.1]
        ),

        barmode="group",

        template="plotly_white",

        height=750,

        legend=dict(
            title="Model",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0
        ),

        updatemenus=[
            dict(
                buttons=buttons,
                direction="down",
                showactive=True,
                x=1,
                xanchor="right",
                y=1.12,
                yanchor="top"
            )
        ],

        margin=dict(
            l=70,
            r=40,
            t=120,
            b=120
        )
    )

    return fig


In [96]:
experiments = {
    "XLM-R Base": Path(
        r"C:\Users\alrazz\Downloads\Results\_MultiCore_XLMR_base_evaluation"
    ),

    "BGE-M3 n512 TH035": Path(
        r"C:\Users\alrazz\Downloads\Results\_MultiCore_BGE_m3_base_evaluation_n512_TH035"
    ),

    "BGE-M3 Base": Path(
        r"C:\Users\alrazz\Downloads\Results\_MultiCore_BGE_m3_base_evaluation"
    ),
}


In [97]:
figures = {}

for experiment_name, data_dir in experiments.items():

    print(f"Loading: {experiment_name}")

    combined = load_experiment(data_dir)

    figures[experiment_name] = create_classification_plot(
        combined,
        experiment_name
    )

    print(f"  Loaded {len(combined)} rows")


Loading: XLM-R Base
  Loaded 100 rows
Loading: BGE-M3 n512 TH035
  Loaded 100 rows
Loading: BGE-M3 Base
  Loaded 100 rows


In [100]:
for name, fig in figures.items():

    print("=" * 60)
    print(name)
    print("=" * 60)

    fig.show()


XLM-R Base
BGE-M3 n512 TH035
BGE-M3 Base


## All together

In [101]:
experiments = {
    "XLM-R Base": Path(
        r"C:\Users\alrazz\Downloads\Results\_MultiCore_XLMR_base_evaluation"
    ),

    "BGE-M3 n512 TH035": Path(
        r"C:\Users\alrazz\Downloads\Results\_MultiCore_BGE_m3_base_evaluation_n512_TH035"
    ),

    "BGE-M3 Base": Path(
        r"C:\Users\alrazz\Downloads\Results\_MultiCore_BGE_m3_base_evaluation"
    ),
}


In [102]:
all_data = []

for experiment_name, data_dir in experiments.items():

    combined = load_experiment(data_dir)

    combined["experiment"] = experiment_name

    all_data.append(combined)


all_combined = pd.concat(
    all_data,
    ignore_index=True
)

display(all_combined.head())


,variation,label,precision,recall,f1-score,support,experiment
0,Hybrid,MT,0.818182,0.290323,0.428571,31.0,XLM-R Base
1,Hybrid,LY,0.862069,0.490196,0.625000,51.0,XLM-R Base
2,Hybrid,SP,0.893617,0.688525,0.777778,61.0,XLM-R Base
3,Hybrid,ID,0.222222,0.137931,0.170213,58.0,XLM-R Base
4,Hybrid,NA,0.835443,0.680412,0.750000,194.0,XLM-R Base


In [103]:
def create_class_grouped_plot(all_combined, metric="f1-score"):

    # Create a configuration name
    all_combined = all_combined.copy()

    all_combined["configuration"] = (
        all_combined["experiment"]
        + " — "
        + all_combined["variation"]
    )

    # Order of experiments
    experiment_order = [
        "XLM-R Base",
        "BGE-M3 n512 TH035",
        "BGE-M3 Base",
    ]

    # Order of variations
    variation_order = [
        "Hybrid",
        "ID Hybrid",
        "Single",
        "SP Hybrid",
    ]

    # Order classes according to their appearance
    label_order = (
        all_combined["label"]
        .drop_duplicates()
        .tolist()
    )

    # -----------------------------------------------------
    # Create figure
    # -----------------------------------------------------

    fig = go.Figure()

    # One trace for each of the 12 configurations
    for experiment in experiment_order:

        for variation in variation_order:

            df = all_combined[
                (all_combined["experiment"] == experiment)
                & (all_combined["variation"] == variation)
            ]

            # Make sure labels appear in the desired order
            df = (
                df.set_index("label")
                .reindex(label_order)
                .reset_index()
            )

            configuration = f"{experiment} — {variation}"

            fig.add_trace(
                go.Bar(
                    x=df["label"],
                    y=df[metric],
                    name=configuration,

                    hovertemplate=(
                        "<b>Class:</b> %{x}<br>"
                        f"<b>Experiment:</b> {experiment}<br>"
                        f"<b>Variation:</b> {variation}<br>"
                        f"<b>{metric}:</b> %{{y:.3f}}"
                        "<extra></extra>"
                    ),
                )
            )

    # -----------------------------------------------------
    # Layout
    # -----------------------------------------------------

    fig.update_layout(
        title=f"{metric.title()} Comparison by Class",

        xaxis=dict(
            title="Class",
            categoryorder="array",
            categoryarray=label_order,
        ),

        yaxis=dict(
            title=metric.title(),
            range=[0, 1.1],
        ),

        barmode="group",

        template="plotly_white",

        height=800,

        legend=dict(
            title="Configuration",
        ),

        margin=dict(
            l=70,
            r=40,
            t=80,
            b=120,
        ),
    )

    return fig


In [104]:
fig = create_class_grouped_plot(
    all_combined,
    metric="f1-score"
)

fig.show()


In [105]:
def create_class_grouped_plot(all_combined, metric="f1-score"):

    all_combined = all_combined.copy()

    all_combined["configuration"] = (
        all_combined["experiment"]
        + " — "
        + all_combined["variation"]
    )

    experiment_order = [
        "XLM-R Base",
        "BGE-M3 n512 TH035",
        "BGE-M3 Base",
    ]

    variation_order = [
        "Hybrid",
        "ID Hybrid",
        "Single",
        "SP Hybrid",
    ]

    label_order = (
        all_combined["label"]
        .drop_duplicates()
        .tolist()
    )

    fig = go.Figure()

    # -----------------------------------------------------
    # Colors
    # -----------------------------------------------------

    colors = {
        "XLM-R Base": "#636EFA",
        "BGE-M3 n512 TH035": "#EF553B",
        "BGE-M3 Base": "#00CC96",
    }

    # -----------------------------------------------------
    # Add bars
    # -----------------------------------------------------

    for experiment in experiment_order:

        for variation in variation_order:

            df = all_combined[
                (all_combined["experiment"] == experiment)
                & (all_combined["variation"] == variation)
            ]

            df = (
                df.set_index("label")
                .reindex(label_order)
                .reset_index()
            )

            fig.add_trace(
                go.Bar(
                    x=df["label"],
                    y=df[metric],

                    name=f"{experiment} — {variation}",

                    marker_color=colors[experiment],

                    hovertemplate=(
                        "<b>Class:</b> %{x}<br>"
                        f"<b>Model:</b> {experiment}<br>"
                        f"<b>Variation:</b> {variation}<br>"
                        f"<b>{metric}:</b> %{{y:.3f}}"
                        "<extra></extra>"
                    ),
                )
            )

    # -----------------------------------------------------
    # Add vertical separators
    # -----------------------------------------------------

    # There are 4 variations per experiment.
    # Add a separator after each group of 4 bars.

    for i, label in enumerate(label_order):

        # Position of the class category
        x_position = i

        # These positions are approximate because
        # the bars are grouped around each category.

        fig.add_vline(
            x=x_position + 0.5,
            line_width=1,
            line_dash="dot",
            line_color="gray",
            opacity=0.5,
        )

    # -----------------------------------------------------
    # Layout
    # -----------------------------------------------------

    fig.update_layout(

        title=f"{metric.title()} Comparison by Class",

        xaxis=dict(
            title="Class",
            categoryorder="array",
            categoryarray=label_order,
        ),

        yaxis=dict(
            title=metric.title(),
            range=[0, 1.1],
        ),

        barmode="group",

        template="plotly_white",

        height=800,

        legend=dict(
            title="Configuration",
        ),

        margin=dict(
            l=70,
            r=40,
            t=80,
            b=120,
        ),
    )

    return fig


In [106]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go


def create_separated_plot(all_combined, metric="f1-score"):

    experiments = [
        "XLM-R Base",
        "BGE-M3 n512 TH035",
        "BGE-M3 Base",
    ]

    variations = [
        "Hybrid",
        "ID Hybrid",
        "Single",
        "SP Hybrid",
    ]

    colors = {
        "Hybrid": "#636EFA",
        "ID Hybrid": "#EF553B",
        "Single": "#00CC96",
        "SP Hybrid": "#AB63FA",
    }

    # Preserve class order
    labels = (
        all_combined["label"]
        .drop_duplicates()
        .tolist()
    )

    # -----------------------------------------------------
    # Create 3 vertically stacked subplots
    # -----------------------------------------------------

    fig = make_subplots(
        rows=3,
        cols=1,

        shared_xaxes=True,

        vertical_spacing=0.08,

        subplot_titles=experiments
    )

    # -----------------------------------------------------
    # Add bars
    # -----------------------------------------------------

    for row, experiment in enumerate(experiments, start=1):

        for variation in variations:

            df = all_combined[
                (all_combined["experiment"] == experiment)
                & (all_combined["variation"] == variation)
            ]

            df = (
                df.set_index("label")
                .reindex(labels)
                .reset_index()
            )

            fig.add_trace(

                go.Bar(
                    x=df["label"],
                    y=df[metric],

                    name=variation,

                    marker_color=colors[variation],

                    legendgroup=variation,

                    showlegend=(row == 1),

                    text=df[metric].round(2),
                    textposition="outside",

                    hovertemplate=(
                        "<b>Class:</b> %{x}<br>"
                        f"<b>Model:</b> {experiment}<br>"
                        f"<b>Variation:</b> {variation}<br>"
                        f"<b>{metric}:</b> %{{y:.3f}}"
                        "<extra></extra>"
                    ),
                ),

                row=row,
                col=1
            )

        # Y-axis for each subplot
        fig.update_yaxes(
            range=[0, 1.1],
            title_text=metric.title(),
            row=row,
            col=1
        )

    # -----------------------------------------------------
    # Layout
    # -----------------------------------------------------

    fig.update_layout(

        title=f"{metric.title()} Comparison by Model and Class",

        barmode="group",

        template="plotly_white",

        height=1100,

        legend=dict(
            title="Variation",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0
        ),

        margin=dict(
            l=80,
            r=40,
            t=150,
            b=100
        )
    )

    fig.update_xaxes(
        categoryorder="array",
        categoryarray=labels,
        tickangle=-45
    )

    return fig


In [107]:
fig = create_separated_plot(
    all_combined,
    metric="f1-score"
)

fig.show()


## based on class on each row

In [108]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go


def create_class_subplots(all_combined, metric="f1-score", cols=3):

    # -----------------------------------------------------
    # Configuration
    # -----------------------------------------------------

    experiments = [
        "XLM-R Base",
        "BGE-M3 n512 TH035",
        "BGE-M3 Base",
    ]

    variations = [
        "Hybrid",
        "ID Hybrid",
        "Single",
        "SP Hybrid",
    ]

    # Colors represent the variation
    colors = {
        "Hybrid": "#636EFA",
        "ID Hybrid": "#EF553B",
        "Single": "#00CC96",
        "SP Hybrid": "#AB63FA",
    }

    # -----------------------------------------------------
    # Get classes
    # -----------------------------------------------------

    labels = (
        all_combined["label"]
        .drop_duplicates()
        .tolist()
    )

    n_classes = len(labels)

    rows = (n_classes + cols - 1) // cols

    # -----------------------------------------------------
    # Create subplots
    # -----------------------------------------------------

    fig = make_subplots(
        rows=rows,
        cols=cols,

        subplot_titles=labels,

        horizontal_spacing=0.05,
        vertical_spacing=0.10,
    )

    # -----------------------------------------------------
    # Add bars
    # -----------------------------------------------------

    for class_index, label in enumerate(labels):

        row = class_index // cols + 1
        col = class_index % cols + 1

        # Each subplot contains 12 bars
        # 3 models × 4 variations

        x_values = []
        y_values = []
        variation_values = []
        experiment_values = []

        for experiment in experiments:

            for variation in variations:

                df = all_combined[
                    (all_combined["experiment"] == experiment)
                    & (all_combined["variation"] == variation)
                    & (all_combined["label"] == label)
                ]

                if df.empty:
                    value = None
                else:
                    value = df.iloc[0][metric]

                # Create x label
                x_values.append(
                    f"{experiment}\n{variation}"
                )

                y_values.append(value)
                variation_values.append(variation)
                experiment_values.append(experiment)

        # -------------------------------------------------
        # Add each variation separately
        # -------------------------------------------------

        for variation in variations:

            x = []
            y = []
            experiments_for_hover = []

            for experiment in experiments:

                df = all_combined[
                    (all_combined["experiment"] == experiment)
                    & (all_combined["variation"] == variation)
                    & (all_combined["label"] == label)
                ]

                if df.empty:
                    value = None
                else:
                    value = df.iloc[0][metric]

                x.append(
                    f"{experiment}\n{variation}"
                )

                y.append(value)
                experiments_for_hover.append(experiment)

            fig.add_trace(

                go.Bar(
                    x=x,
                    y=y,

                    name=variation,

                    marker_color=colors[variation],

                    legendgroup=variation,

                    showlegend=(
                        class_index == 0
                    ),

                    text=[
                        f"{v:.2f}" if v is not None else ""
                        for v in y
                    ],

                    textposition="outside",

                    hovertemplate=(
                        f"<b>Class:</b> {label}<br>"
                        "<b>Model:</b> %{customdata}<br>"
                        f"<b>Variation:</b> {variation}<br>"
                        f"<b>{metric}:</b> %{{y:.3f}}"
                        "<extra></extra>"
                    ),

                    customdata=experiments_for_hover,
                ),

                row=row,
                col=col
            )

        # -------------------------------------------------
        # Y-axis
        # -------------------------------------------------

        fig.update_yaxes(
            range=[0, 1.1],
            row=row,
            col=col,
            title_text=metric.title()
            if col == 1
            else None,
        )

    # -----------------------------------------------------
    # Layout
    # -----------------------------------------------------

    fig.update_layout(

        title=(
            f"{metric.title()} Comparison "
            "by Class, Model and Variation"
        ),

        barmode="group",

        template="plotly_white",

        height=350 * rows,

        legend=dict(
            title="Variation",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0,
        ),

        margin=dict(
            l=70,
            r=40,
            t=130,
            b=100,
        ),
    )

    # -----------------------------------------------------
    # X-axis formatting
    # -----------------------------------------------------

    fig.update_xaxes(
        tickangle=-45,
        tickfont=dict(size=8),
    )

    return fig


In [109]:
fig = create_class_subplots(
    all_combined,
    metric="f1-score",
    cols=3
)

fig.show()


## Maybe even better

In [110]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go


def create_class_subplots(
    all_combined,
    metric="f1-score",
    cols=3
):

    # =====================================================
    # Configuration
    # =====================================================

    experiments = [
        "XLM-R Base",
        "BGE-M3 n512 TH035",
        "BGE-M3 Base",
    ]

    variations = [
        "Hybrid",
        "ID Hybrid",
        "Single",
        "SP Hybrid",
    ]

    colors = {
        "Hybrid": "#636EFA",
        "ID Hybrid": "#EF553B",
        "Single": "#00CC96",
        "SP Hybrid": "#AB63FA",
    }

    # =====================================================
    # Classes
    # =====================================================

    labels = (
        all_combined["label"]
        .drop_duplicates()
        .tolist()
    )

    n_classes = len(labels)

    rows = (n_classes + cols - 1) // cols

    # =====================================================
    # Create subplots
    # =====================================================

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=labels,
        horizontal_spacing=0.06,
        vertical_spacing=0.10,
    )

    # =====================================================
    # Add bars
    # =====================================================

    for class_index, label in enumerate(labels):

        row = class_index // cols + 1
        col = class_index % cols + 1

        # ---------------------------------------------
        # Create x positions explicitly
        # ---------------------------------------------

        x_values = []
        x_labels = []
        y_values = []
        variation_values = []
        model_values = []

        position = 0

        for experiment in experiments:

            for variation in variations:

                df = all_combined[
                    (all_combined["experiment"] == experiment)
                    & (all_combined["variation"] == variation)
                    & (all_combined["label"] == label)
                ]

                if df.empty:
                    value = None
                else:
                    value = df.iloc[0][metric]

                x_values.append(position)
                x_labels.append(variation)

                y_values.append(value)

                variation_values.append(variation)
                model_values.append(experiment)

                position += 1

        # ---------------------------------------------
        # Add bars
        # ---------------------------------------------

        for variation in variations:

            x = []
            y = []
            models = []

            for i in range(len(x_values)):

                if variation_values[i] == variation:

                    x.append(x_values[i])
                    y.append(y_values[i])
                    models.append(model_values[i])

            fig.add_trace(

                go.Bar(
                    x=x,
                    y=y,

                    name=variation,

                    marker_color=colors[variation],

                    legendgroup=variation,

                    showlegend=(
                        class_index == 0
                    ),

                    text=[
                        f"{v:.2f}" if v is not None else ""
                        for v in y
                    ],

                    textposition="outside",

                    customdata=models,

                    hovertemplate=(
                        f"<b>Class:</b> {label}<br>"
                        "<b>Model:</b> %{customdata}<br>"
                        f"<b>Variation:</b> {variation}<br>"
                        f"<b>{metric}:</b> %{{y:.3f}}"
                        "<extra></extra>"
                    ),
                ),

                row=row,
                col=col,
            )

        # =================================================
        # Model separators
        # =================================================

        # Between XLM-R and BGE-n512
        fig.add_vline(
            x=3.5,
            line_width=2,
            line_dash="dash",
            line_color="gray",
            opacity=0.7,
            row=row,
            col=col,
        )

        # Between BGE-n512 and BGE
        fig.add_vline(
            x=7.5,
            line_width=2,
            line_dash="dash",
            line_color="gray",
            opacity=0.7,
            row=row,
            col=col,
        )

        # =================================================
        # Model labels above the bars
        # =================================================

        fig.add_annotation(
            x=1.5,
            y=1.12,
            xref=f"x{class_index + 1}",
            yref=f"y{class_index + 1}",
            text="<b>XLM-R Base</b>",
            showarrow=False,
            font=dict(size=10),
            row=row,
            col=col,
        )

        fig.add_annotation(
            x=5.5,
            y=1.12,
            xref=f"x{class_index + 1}",
            yref=f"y{class_index + 1}",
            text="<b>BGE-M3 n512</b>",
            showarrow=False,
            font=dict(size=10),
            row=row,
            col=col,
        )

        fig.add_annotation(
            x=9.5,
            y=1.12,
            xref=f"x{class_index + 1}",
            yref=f"y{class_index + 1}",
            text="<b>BGE-M3 Base</b>",
            showarrow=False,
            font=dict(size=10),
            row=row,
            col=col,
        )

        # =================================================
        # Y axis
        # =================================================

        fig.update_yaxes(
            range=[0, 1.15],
            row=row,
            col=col,
            title_text=(
                metric.title()
                if col == 1
                else None
            ),
        )

        # =================================================
        # X axis
        # =================================================

        fig.update_xaxes(
            tickmode="array",
            tickvals=list(range(12)),
            ticktext=[
                "H",
                "ID",
                "S",
                "SP",
                "H",
                "ID",
                "S",
                "SP",
                "H",
                "ID",
                "S",
                "SP",
            ],
            row=row,
            col=col,
        )

    # =====================================================
    # Layout
    # =====================================================

    fig.update_layout(

        title=(
            f"{metric.title()} Comparison "
            "by Class, Model and Variation"
        ),

        barmode="group",

        template="plotly_white",

        height=350 * rows,

        legend=dict(
            title="Variation",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0,
        ),

        margin=dict(
            l=70,
            r=40,
            t=150,
            b=80,
        ),
    )

    return fig


In [111]:
fig = create_class_subplots(
    all_combined,
    metric="f1-score",
    cols=3
)

fig.show()


## Maybe even better

In [118]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go


def create_class_subplots(
    all_combined,
    metric="f1-score"
):

    # =====================================================
    # Configuration
    # =====================================================

    experiments = [
        "XLM-R Base",
        "BGE-M3 n512 TH035",
        "BGE-M3 Base",
    ]

    variations = [
        "Hybrid",
        "ID Hybrid",
        "Single",
        "SP Hybrid",
    ]

    colors = {
        "Hybrid": "#636EFA",
        "ID Hybrid": "#EF553B",
        "Single": "#00CC96",
        "SP Hybrid": "#AB63FA",
    }

    # =====================================================
    # Classes
    # =====================================================

    labels = (
        all_combined["label"]
        .drop_duplicates()
        .tolist()
    )

    n_classes = len(labels)

    # IMPORTANT:
    # One class per row
    rows = n_classes
    cols = 1

    # =====================================================
    # Create subplots
    # =====================================================

    fig = make_subplots(
        rows=rows,
        cols=1,

        subplot_titles=labels,

        vertical_spacing=0.025,
    )

    # =====================================================
    # Add bars
    # =====================================================

    for class_index, label in enumerate(labels):

        row = class_index + 1

        # -------------------------------------------------
        # Positions
        #
        # XLM-R       -> 0, 1, 2, 3
        # BGE-n512    -> 5, 6, 7, 8
        # BGE-M3 Base -> 10, 11, 12, 13
        #
        # Therefore:
        #
        # H ID S SP | H ID S SP | H ID S SP
        #            ^           ^
        #         separator   separator
        # -------------------------------------------------

        model_start_positions = {
            "XLM-R Base": 0,
            "BGE-M3 n512 TH035": 5,
            "BGE-M3 Base": 10,
        }

        # -------------------------------------------------
        # Add each variation
        # -------------------------------------------------

        for variation_index, variation in enumerate(variations):

            x = []
            y = []
            models = []

            for experiment in experiments:

                df = all_combined[
                    (all_combined["experiment"] == experiment)
                    & (all_combined["variation"] == variation)
                    & (all_combined["label"] == label)
                ]

                if df.empty:
                    value = None
                else:
                    value = df.iloc[0][metric]

                position = (
                    model_start_positions[experiment]
                    + variation_index
                )

                x.append(position)
                y.append(value)
                models.append(experiment)

            # -------------------------------------------------
            # Add trace
            # -------------------------------------------------

            fig.add_trace(

                go.Bar(
                    x=x,
                    y=y,

                    name=variation,

                    marker_color=colors[variation],

                    # IMPORTANT
                    # Makes the bars fill their positions
                    width=0.9,

                    legendgroup=variation,

                    showlegend=(
                        class_index == 0
                    ),

                    text=[
                        f"{v:.2f}" if v is not None else ""
                        for v in y
                    ],

                    textposition="outside",

                    customdata=models,

                    hovertemplate=(
                        f"<b>Class:</b> {label}<br>"
                        "<b>Model:</b> %{customdata}<br>"
                        f"<b>Variation:</b> {variation}<br>"
                        f"<b>{metric}:</b> %{{y:.3f}}"
                        "<extra></extra>"
                    ),
                ),

                row=row,
                col=1,
            )

        # =================================================
        # Model separators
        # =================================================

        # Between XLM-R and BGE-n512
        fig.add_vline(
            x=4,
            line_width=2,
            line_dash="dash",
            line_color="gray",
            opacity=0.7,

            row=row,
            col=1,
        )

        # Between BGE-n512 and BGE-M3 Base
        fig.add_vline(
            x=9,
            line_width=2,
            line_dash="dash",
            line_color="gray",
            opacity=0.7,

            row=row,
            col=1,
        )

        # =================================================
        # Model labels
        # =================================================

        fig.add_annotation(
            x=1.5,
            y=1.12,

            text="<b>XLM-R Base</b>",

            showarrow=False,

            font=dict(size=11),

            xref=f"x{class_index + 1}",
            yref=f"y{class_index + 1}",

            row=row,
            col=1,
        )

        fig.add_annotation(
            x=6.5,
            y=1.12,

            text="<b>BGE-M3 n512</b>",

            showarrow=False,

            font=dict(size=11),

            xref=f"x{class_index + 1}",
            yref=f"y{class_index + 1}",

            row=row,
            col=1,
        )

        fig.add_annotation(
            x=11.5,
            y=1.12,

            text="<b>BGE-M3 Base</b>",

            showarrow=False,

            font=dict(size=11),

            xref=f"x{class_index + 1}",
            yref=f"y{class_index + 1}",

            row=row,
            col=1,
        )

        # =================================================
        # Y axis
        # =================================================

        fig.update_yaxes(
            range=[0, 1.15],

            title_text=(
                metric.title()
                if class_index == n_classes // 2
                else None
            ),

            row=row,
            col=1,
        )

        # =================================================
        # X axis
        # =================================================

        fig.update_xaxes(
            tickmode="array",

            tickvals=[
                0, 1, 2, 3,
                5, 6, 7, 8,
                10, 11, 12, 13
            ],

            ticktext=[
                "Hybrid",
                "ID Hybrid",
                "Single",
                "SP Hybrid",

                "Hybrid",
                "ID Hybrid",
                "Single",
                "SP Hybrid",

                "Hybrid",
                "ID Hybrid",
                "Single",
                "SP Hybrid",
            ],

            tickangle=-45,

            range=[-0.8, 13.8],

            row=row,
            col=1,
        )

    # =====================================================
    # Layout
    # =====================================================

    fig.update_layout(

        title=(
            f"{metric.title()} Comparison "
            "by Class, Model and Variation"
        ),

        barmode="overlay",

        template="plotly_white",

        # One class per row
        height=280 * rows,

        legend=dict(
            title="Variation",

            orientation="h",

            yanchor="bottom",
            y=1.02,

            xanchor="left",
            x=0,
        ),

        margin=dict(
            l=80,
            r=40,
            t=140,
            b=100,
        ),
    )

    return fig


In [113]:
fig = create_class_subplots(
    all_combined,
    metric="f1-score"
)

fig.show()


## This Work

This is the best I have got use this

In [120]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go


def create_class_subplots(
    all_combined,
    metric="f1-score"
):

    # =====================================================
    # Configuration
    # =====================================================

    experiments = [
        "XLM-R Base",
        "BGE-M3 n512 TH035",
        "BGE-M3 Base",
    ]

    variations = [
        "Hybrid",
        "ID Hybrid",
        "Single",
        "SP Hybrid",
    ]

    colors = {
        "Hybrid": "#636EFA",
        "ID Hybrid": "#EF553B",
        "Single": "#00CC96",
        "SP Hybrid": "#AB63FA",
    }

    # =====================================================
    # Classes
    # =====================================================

    labels = (
        all_combined["label"]
        .drop_duplicates()
        .tolist()
    )

    n_classes = len(labels)

    # One class per row
    rows = n_classes
    cols = 1

    # =====================================================
    # Create subplots
    # =====================================================

    fig = make_subplots(
        rows=rows,
        cols=1,
        subplot_titles=labels,
        vertical_spacing=0.025,
    )

    # =====================================================
    # Positions of the three model groups
    # =====================================================

    model_start_positions = {
        "XLM-R Base": 0,
        "BGE-M3 n512 TH035": 5,
        "BGE-M3 Base": 10,
    }

    # =====================================================
    # Add bars
    # =====================================================

    for class_index, label in enumerate(labels):

        row = class_index + 1

        # -------------------------------------------------
        # Add each variation
        # -------------------------------------------------

        for variation_index, variation in enumerate(variations):

            x = []
            y = []
            models = []

            for experiment in experiments:

                df = all_combined[
                    (all_combined["experiment"] == experiment)
                    & (all_combined["variation"] == variation)
                    & (all_combined["label"] == label)
                ]

                if df.empty:
                    value = None
                else:
                    value = df.iloc[0][metric]

                position = (
                    model_start_positions[experiment]
                    + variation_index
                )

                x.append(position)
                y.append(value)
                models.append(experiment)

            # -------------------------------------------------
            # Add trace
            # -------------------------------------------------

            fig.add_trace(

                go.Bar(
                    x=x,
                    y=y,

                    name=variation,

                    marker_color=colors[variation],

                    # =================================================
                    # IMPORTANT:
                    # width = 1 means adjacent bars touch each other
                    # =================================================
                    width=1.0,

                    marker_line=dict(
                        width=0.5,
                        color="white",
                    ),

                    legendgroup=variation,

                    showlegend=(
                        class_index == 0
                    ),

                    text=[
                        f"{v:.2f}" if v is not None else ""
                        for v in y
                    ],

                    textposition="outside",

                    customdata=models,

                    hovertemplate=(
                        f"<b>Class:</b> {label}<br>"
                        "<b>Model:</b> %{customdata}<br>"
                        f"<b>Variation:</b> {variation}<br>"
                        f"<b>{metric}:</b> %{{y:.3f}}"
                        "<extra></extra>"
                    ),
                ),

                row=row,
                col=1,
            )

        # =================================================
        # Model separators
        # =================================================

        # Between XLM-R and BGE-M3 n512
        fig.add_vline(
            x=4,
            line_width=2,
            line_dash="dash",
            line_color="gray",
            opacity=0.7,
            row=row,
            col=1,
        )

        # Between BGE-M3 n512 and BGE-M3 Base
        fig.add_vline(
            x=9,
            line_width=2,
            line_dash="dash",
            line_color="gray",
            opacity=0.7,
            row=row,
            col=1,
        )

        # =================================================
        # Model labels
        # =================================================

        fig.add_annotation(
            x=1.5,
            y=1.12,
            text="<b>XLM-R Base</b>",
            showarrow=False,
            font=dict(size=11),
            xref=f"x{class_index + 1}",
            yref=f"y{class_index + 1}",
            row=row,
            col=1,
        )

        fig.add_annotation(
            x=6.5,
            y=1.12,
            text="<b>BGE-M3 n512</b>",
            showarrow=False,
            font=dict(size=11),
            xref=f"x{class_index + 1}",
            yref=f"y{class_index + 1}",
            row=row,
            col=1,
        )

        fig.add_annotation(
            x=11.5,
            y=1.12,
            text="<b>BGE-M3 Base</b>",
            showarrow=False,
            font=dict(size=11),
            xref=f"x{class_index + 1}",
            yref=f"y{class_index + 1}",
            row=row,
            col=1,
        )

        # =================================================
        # Y axis
        # =================================================

        fig.update_yaxes(
            range=[0, 1.15],

            title_text=(
                metric.title()
                if class_index == n_classes // 2
                else None
            ),

            row=row,
            col=1,
        )

        # =================================================
        # X axis
        # =================================================

        fig.update_xaxes(

            tickmode="array",

            tickvals=[
                0, 1, 2, 3,
                5, 6, 7, 8,
                10, 11, 12, 13
            ],

            ticktext=[
                "Hybrid",
                "ID Hybrid",
                "Single",
                "SP Hybrid",

                "Hybrid",
                "ID Hybrid",
                "Single",
                "SP Hybrid",

                "Hybrid",
                "ID Hybrid",
                "Single",
                "SP Hybrid",
            ],

            tickangle=-45,

            range=[
                -0.5,
                13.5
            ],

            row=row,
            col=1,
        )

    # =====================================================
    # Layout
    # =====================================================

    fig.update_layout(

        title=(
            f"{metric.title()} Comparison "
            "by Class, Model and Variation"
        ),

        # We explicitly control the x positions,
        # so overlay is appropriate here.
        barmode="overlay",

        template="plotly_white",

        # One class per row
        height=280 * rows,

        legend=dict(
            title="Variation",

            orientation="h",

            yanchor="bottom",
            y=1.02,

            xanchor="left",
            x=0,
        ),

        margin=dict(
            l=80,
            r=40,
            t=140,
            b=100,
        ),
    )

    return fig


In [121]:
fig = create_class_subplots(
    all_combined,
    metric="f1-score"
)

fig.show()


## Add others

In [125]:
from pathlib import Path
import pandas as pd


experiments = {
    # Existing experiments
    "XLM-R Base": Path(
        r"C:\Users\alrazz\Downloads\Results\_MultiCore_XLMR_base_evaluation"
    ),

    "BGE-M3 n512 TH035": Path(
        r"C:\Users\alrazz\Downloads\Results\_MultiCore_BGE_m3_base_evaluation_n512_TH035"
    ),

    "BGE-M3 Base": Path(
        r"C:\Users\alrazz\Downloads\Results\_MultiCore_BGE_m3_base_evaluation"
    ),

    # New fine-tuned experiments
    "XLM-R Fine-tuned": Path(
        r"C:\Users\alrazz\Downloads\Results\XLMR_Finetuned_Threshold_tuning\_test_evaluation"
    ),

    "BGE-M3 Fine-tuned": Path(
        r"C:\Users\alrazz\Downloads\Results\BGEM3_Finetuned_Threshold_tuning\_test_evaluation"
    ),
}


# Map folder names to the variation names used by your plotting code
variation_folders = {
    "Combined_hybrid_no_NA": "Hybrid",
    "Combined_ID_hybrid_no_NA": "ID Hybrid",
    "Combined_single_no_NA": "Single",
    "Combined_SP_hybrid_no_NA": "SP Hybrid",
}


all_data = []


# =========================================================
# Existing experiments
# =========================================================

existing_experiments = {
    "XLM-R Base": experiments["XLM-R Base"],
    "BGE-M3 n512 TH035": experiments["BGE-M3 n512 TH035"],
    "BGE-M3 Base": experiments["BGE-M3 Base"],
}

for experiment_name, data_dir in existing_experiments.items():

    combined = load_experiment(data_dir)

    combined["experiment"] = experiment_name

    all_data.append(combined)


# =========================================================
# Fine-tuned experiments
# =========================================================

finetuned_experiments = {
    "XLM-R Fine-tuned": experiments["XLM-R Fine-tuned"],
    "BGE-M3 Fine-tuned": experiments["BGE-M3 Fine-tuned"],
}


for experiment_name, test_evaluation_dir in finetuned_experiments.items():

    for folder_name, variation in variation_folders.items():

        csv_path = (
            test_evaluation_dir
            / folder_name
            / "test_classification_report_per_class.csv"
        )

        if not csv_path.exists():
            print(f"WARNING: File not found: {csv_path}")
            continue

        df = pd.read_csv(csv_path, index_col=0, keep_default_na=False)

        # Convert the class index into a normal column
        df = df.reset_index()

        # Rename first column to "label"
        df = df.rename(columns={df.columns[0]: "label"})

        # Add variation and experiment information
        df["variation"] = variation
        df["experiment"] = experiment_name

        all_data.append(df)


# =========================================================
# Combine everything
# =========================================================

all_combined = pd.concat(
    all_data,
    ignore_index=True
)

display(all_combined.head())
display(all_combined["experiment"].value_counts())
display(all_combined["variation"].value_counts())


,variation,label,precision,recall,f1-score,support,experiment
0,Hybrid,MT,0.818182,0.290323,0.428571,31.0,XLM-R Base
1,Hybrid,LY,0.862069,0.490196,0.625000,51.0,XLM-R Base
2,Hybrid,SP,0.893617,0.688525,0.777778,61.0,XLM-R Base
3,Hybrid,ID,0.222222,0.137931,0.170213,58.0,XLM-R Base
4,Hybrid,NA,0.835443,0.680412,0.750000,194.0,XLM-R Base


experiment
BGE-M3 Fine-tuned    116
XLM-R Fine-tuned     116
XLM-R Base           100
BGE-M3 Base          100
BGE-M3 n512 TH035    100
Name: count, dtype: int64

variation
Hybrid       133
ID Hybrid    133
Single       133
SP Hybrid    133
Name: count, dtype: int64

In [126]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go


def create_class_subplots(
    all_combined,
    metric="f1-score"
):

    # =====================================================
    # Configuration
    # =====================================================

    experiments = [
        "XLM-R Base",
        "BGE-M3 n512 TH035",
        "BGE-M3 Base",
        "XLM-R Fine-tuned",
        "BGE-M3 Fine-tuned",
    ]

    variations = [
        "Hybrid",
        "ID Hybrid",
        "Single",
        "SP Hybrid",
    ]

    colors = {
        "Hybrid": "#636EFA",
        "ID Hybrid": "#EF553B",
        "Single": "#00CC96",
        "SP Hybrid": "#AB63FA",
    }

    # =====================================================
    # Classes
    # =====================================================

    labels = (
        all_combined["label"]
        .drop_duplicates()
        .tolist()
    )

    # Remove average rows if they somehow appear
    labels = [
        label for label in labels
        if label not in [
            "micro avg",
            "macro avg",
            "weighted avg",
            "samples avg",
        ]
    ]

    n_classes = len(labels)

    # =====================================================
    # Create subplots
    # =====================================================

    fig = make_subplots(
        rows=n_classes,
        cols=1,
        subplot_titles=labels,
        vertical_spacing=0.025,
    )

    # =====================================================
    # Positions of the five model groups
    # =====================================================

    model_start_positions = {
        "XLM-R Base": 0,
        "BGE-M3 n512 TH035": 5,
        "BGE-M3 Base": 10,
        "XLM-R Fine-tuned": 15,
        "BGE-M3 Fine-tuned": 20,
    }

    # =====================================================
    # Add bars
    # =====================================================

    for class_index, label in enumerate(labels):

        row = class_index + 1

        # -------------------------------------------------
        # Add each variation
        # -------------------------------------------------

        for variation_index, variation in enumerate(variations):

            x = []
            y = []
            models = []

            for experiment in experiments:

                df = all_combined[
                    (all_combined["experiment"] == experiment)
                    & (all_combined["variation"] == variation)
                    & (all_combined["label"] == label)
                ]

                if df.empty:
                    value = None
                else:
                    value = df.iloc[0][metric]

                position = (
                    model_start_positions[experiment]
                    + variation_index
                )

                x.append(position)
                y.append(value)
                models.append(experiment)

            # -------------------------------------------------
            # Add trace
            # -------------------------------------------------

            fig.add_trace(

                go.Bar(
                    x=x,
                    y=y,

                    name=variation,

                    marker_color=colors[variation],

                    width=1.0,

                    marker_line=dict(
                        width=0.5,
                        color="white",
                    ),

                    legendgroup=variation,

                    showlegend=(
                        class_index == 0
                    ),

                    text=[
                        f"{v:.2f}" if v is not None else ""
                        for v in y
                    ],

                    textposition="outside",

                    customdata=models,

                    hovertemplate=(
                        f"<b>Class:</b> {label}<br>"
                        "<b>Model:</b> %{customdata}<br>"
                        f"<b>Variation:</b> {variation}<br>"
                        f"<b>{metric}:</b> %{{y:.3f}}"
                        "<extra></extra>"
                    ),
                ),

                row=row,
                col=1,
            )

        # =================================================
        # Model separators
        # =================================================

        separators = [
            4,
            9,
            14,
            19,
        ]

        for separator in separators:

            fig.add_vline(
                x=separator,
                line_width=2,
                line_dash="dash",
                line_color="gray",
                opacity=0.7,
                row=row,
                col=1,
            )

        # =================================================
        # Model labels
        # =================================================

        model_annotations = [
            (1.5, "XLM-R Base"),
            (6.5, "BGE-M3 n512"),
            (11.5, "BGE-M3 Base"),
            (16.5, "XLM-R Fine-tuned"),
            (21.5, "BGE-M3 Fine-tuned"),
        ]

        for x_position, model_name in model_annotations:

            fig.add_annotation(
                x=x_position,
                y=1.12,

                text=f"<b>{model_name}</b>",

                showarrow=False,

                font=dict(size=11),

                xref=f"x{class_index + 1}",
                yref=f"y{class_index + 1}",

                row=row,
                col=1,
            )

        # =================================================
        # Y axis
        # =================================================

        fig.update_yaxes(

            range=[0, 1.15],

            title_text=(
                metric.title()
                if class_index == n_classes // 2
                else None
            ),

            row=row,
            col=1,
        )

        # =================================================
        # X axis
        # =================================================

        tickvals = [
            0, 1, 2, 3,
            5, 6, 7, 8,
            10, 11, 12, 13,
            15, 16, 17, 18,
            20, 21, 22, 23,
        ]

        ticktext = (
            variations
            + variations
            + variations
            + variations
            + variations
        )

        fig.update_xaxes(

            tickmode="array",

            tickvals=tickvals,

            ticktext=ticktext,

            tickangle=-45,

            range=[
                -0.5,
                23.5
            ],

            row=row,
            col=1,
        )

    # =====================================================
    # Layout
    # =====================================================

    fig.update_layout(

        title=(
            f"{metric.title()} Comparison "
            "by Class, Model and Variation"
        ),

        barmode="overlay",

        template="plotly_white",

        height=280 * n_classes,

        legend=dict(
            title="Variation",

            orientation="h",

            yanchor="bottom",
            y=1.02,

            xanchor="left",
            x=0,
        ),

        margin=dict(
            l=80,
            r=40,
            t=140,
            b=100,
        ),
    )

    return fig


In [ ]:
fig = create_class_subplots(
    all_combined,
    metric="f1-score"
)

fig.show()


# Complete sub_plots

In [132]:
from pathlib import Path
import pandas as pd


# =========================================================
# Experiment directories
# =========================================================

experiments = {
    # Existing experiments
    "XLM-R Base": Path(
        r"C:\Users\alrazz\Downloads\Results\_MultiCore_XLMR_base_evaluation"
    ),

    "BGE-M3 n512 TH035": Path(
        r"C:\Users\alrazz\Downloads\Results\_MultiCore_BGE_m3_base_evaluation_n512_TH035"
    ),

    "BGE-M3 Base": Path(
        r"C:\Users\alrazz\Downloads\Results\_MultiCore_BGE_m3_base_evaluation"
    ),

    # Fine-tuned experiments
    "XLM-R Fine-tuned": Path(
        r"C:\Users\alrazz\Downloads\Results\XLMR_Finetuned_Threshold_tuning\_test_evaluation"
    ),

    "BGE-M3 Fine-tuned": Path(
        r"C:\Users\alrazz\Downloads\Results\BGEM3_Finetuned_Threshold_tuning\_test_evaluation"
    ),

    # BGE-M3 Fine-tuned Sliding Threshold
    "BGE-M3 Fine-tuned Sliding": Path(
        r"C:\Users\alrazz\Downloads\Results\BGEM3_Finetuned_Sliding_Threshold_tuning\_test_evaluation"
    ),
}


# =========================================================
# Variation folder mapping
# =========================================================

variation_folders = {
    "Combined_hybrid_no_NA": "Hybrid",
    "Combined_ID_hybrid_no_NA": "ID Hybrid",
    "Combined_single_no_NA": "Single",
    "Combined_SP_hybrid_no_NA": "SP Hybrid",
}


all_data = []


# =========================================================
# Existing experiments
# =========================================================

existing_experiments = {
    "XLM-R Base": experiments["XLM-R Base"],
    "BGE-M3 n512 TH035": experiments["BGE-M3 n512 TH035"],
    "BGE-M3 Base": experiments["BGE-M3 Base"],
}


for experiment_name, data_dir in existing_experiments.items():

    combined = load_experiment(data_dir)

    combined["experiment"] = experiment_name

    all_data.append(combined)


# =========================================================
# Fine-tuned experiments
#
# These use:
# test_classification_report_per_class.csv
# =========================================================

finetuned_experiments = {
    "XLM-R Fine-tuned": experiments["XLM-R Fine-tuned"],
    "BGE-M3 Fine-tuned": experiments["BGE-M3 Fine-tuned"],
}


for experiment_name, test_evaluation_dir in finetuned_experiments.items():

    for folder_name, variation in variation_folders.items():

        csv_path = (
            test_evaluation_dir
            / folder_name
            / "test_classification_report_per_class.csv"
        )

        if not csv_path.exists():
            print(f"WARNING: File not found: {csv_path}")
            continue

        df = pd.read_csv(csv_path, index_col=0, keep_default_na=False)

        # Convert index to normal column
        df = df.reset_index()

        # Rename class column
        df = df.rename(columns={df.columns[0]: "label"})

        # Add metadata
        df["variation"] = variation
        df["experiment"] = experiment_name

        all_data.append(df)


# =========================================================
# BGE-M3 Fine-tuned Sliding Threshold
#
# This experiment uses:
# experiment_A_official_test_classification_report.csv
# =========================================================

sliding_experiment_name = "BGE-M3 Fine-tuned Sliding"

sliding_test_evaluation_dir = experiments[
    sliding_experiment_name
]


for folder_name, variation in variation_folders.items():

    csv_path = (
        sliding_test_evaluation_dir
        / folder_name
        / "experiment_A_official_test_classification_report.csv"
    )

    if not csv_path.exists():
        print(f"WARNING: File not found: {csv_path}")
        continue

    df = pd.read_csv(csv_path, index_col=0, keep_default_na=False)

    # Convert index to normal column
    df = df.reset_index()

    # Rename class column
    df = df.rename(columns={df.columns[0]: "label"})

    # Add metadata
    df["variation"] = variation
    df["experiment"] = sliding_experiment_name

    all_data.append(df)


# =========================================================
# Combine everything
# =========================================================

all_combined = pd.concat(
    all_data,
    ignore_index=True
)


# =========================================================
# Check the loaded experiments
# =========================================================

print("\nExperiments:")
print(all_combined["experiment"].value_counts())

print("\nVariations:")
print(all_combined["variation"].value_counts())

display(all_combined.head())



Experiments:
experiment
BGE-M3 Fine-tuned            116
XLM-R Fine-tuned             116
BGE-M3 Fine-tuned Sliding    116
XLM-R Base                   100
BGE-M3 n512 TH035            100
BGE-M3 Base                  100
Name: count, dtype: int64

Variations:
variation
Hybrid       162
ID Hybrid    162
Single       162
SP Hybrid    162
Name: count, dtype: int64


,variation,label,precision,recall,f1-score,support,experiment
0,Hybrid,MT,0.818182,0.290323,0.428571,31.0,XLM-R Base
1,Hybrid,LY,0.862069,0.490196,0.625000,51.0,XLM-R Base
2,Hybrid,SP,0.893617,0.688525,0.777778,61.0,XLM-R Base
3,Hybrid,ID,0.222222,0.137931,0.170213,58.0,XLM-R Base
4,Hybrid,NA,0.835443,0.680412,0.750000,194.0,XLM-R Base


In [133]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go


def create_class_subplots(
    all_combined,
    metric="f1-score"
):

    # =====================================================
    # Configuration
    # =====================================================

    experiments = [
        "XLM-R Base",
        "BGE-M3 n512 TH035",
        "BGE-M3 Base",
        "XLM-R Fine-tuned",
        "BGE-M3 Fine-tuned",
        "BGE-M3 Fine-tuned Sliding",
    ]

    variations = [
        "Hybrid",
        "ID Hybrid",
        "Single",
        "SP Hybrid",
    ]

    colors = {
        "Hybrid": "#636EFA",
        "ID Hybrid": "#EF553B",
        "Single": "#00CC96",
        "SP Hybrid": "#AB63FA",
    }

    # =====================================================
    # Classes
    # =====================================================

    labels = (
        all_combined["label"]
        .drop_duplicates()
        .tolist()
    )

    # Remove average rows
    labels = [
        label for label in labels
        if label not in [
            "micro avg",
            "macro avg",
            "weighted avg",
            "samples avg",
        ]
    ]

    n_classes = len(labels)

    # =====================================================
    # Create subplots
    # =====================================================

    fig = make_subplots(
        rows=n_classes,
        cols=1,
        subplot_titles=labels,
        vertical_spacing=0.025,
    )

    # =====================================================
    # Positions of the six model groups
    # =====================================================

    model_start_positions = {
        "XLM-R Base": 0,
        "BGE-M3 n512 TH035": 5,
        "BGE-M3 Base": 10,
        "XLM-R Fine-tuned": 15,
        "BGE-M3 Fine-tuned": 20,
        "BGE-M3 Fine-tuned Sliding": 25,
    }

    # =====================================================
    # Add bars
    # =====================================================

    for class_index, label in enumerate(labels):

        row = class_index + 1

        # -------------------------------------------------
        # Add each variation
        # -------------------------------------------------

        for variation_index, variation in enumerate(variations):

            x = []
            y = []
            models = []

            for experiment in experiments:

                df = all_combined[
                    (all_combined["experiment"] == experiment)
                    & (all_combined["variation"] == variation)
                    & (all_combined["label"] == label)
                ]

                if df.empty:
                    value = None
                else:
                    value = df.iloc[0][metric]

                position = (
                    model_start_positions[experiment]
                    + variation_index
                )

                x.append(position)
                y.append(value)
                models.append(experiment)

            # -------------------------------------------------
            # Add trace
            # -------------------------------------------------

            fig.add_trace(

                go.Bar(
                    x=x,
                    y=y,

                    name=variation,

                    marker_color=colors[variation],

                    width=1.0,

                    marker_line=dict(
                        width=0.5,
                        color="white",
                    ),

                    legendgroup=variation,

                    showlegend=(
                        class_index == 0
                    ),

                    text=[
                        f"{v:.2f}" if v is not None else ""
                        for v in y
                    ],

                    textposition="outside",

                    customdata=models,

                    hovertemplate=(
                        f"<b>Class:</b> {label}<br>"
                        "<b>Model:</b> %{customdata}<br>"
                        f"<b>Variation:</b> {variation}<br>"
                        f"<b>{metric}:</b> %{{y:.3f}}"
                        "<extra></extra>"
                    ),
                ),

                row=row,
                col=1,
            )

        # =================================================
        # Model separators
        # =================================================

        separators = [
            4,
            9,
            14,
            19,
            24,
        ]

        for separator in separators:

            fig.add_vline(
                x=separator,
                line_width=2,
                line_dash="dash",
                line_color="gray",
                opacity=0.7,
                row=row,
                col=1,
            )

        # =================================================
        # Model labels
        # =================================================

        model_annotations = [
            (1.5, "XLM-R Base"),
            (6.5, "BGE-M3 n512"),
            (11.5, "BGE-M3 Base"),
            (16.5, "XLM-R Fine-tuned"),
            (21.5, "BGE-M3 Fine-tuned"),
            (26.5, "BGE-M3 Fine-tuned Sliding"),
        ]

        for x_position, model_name in model_annotations:

            fig.add_annotation(
                x=x_position,
                y=1.12,

                text=f"<b>{model_name}</b>",

                showarrow=False,

                font=dict(size=11),

                xref=f"x{class_index + 1}",
                yref=f"y{class_index + 1}",

                row=row,
                col=1,
            )

        # =================================================
        # Y axis
        # =================================================

        fig.update_yaxes(

            range=[0, 1.15],

            title_text=(
                metric.title()
                if class_index == n_classes // 2
                else None
            ),

            row=row,
            col=1,
        )

        # =================================================
        # X axis
        # =================================================

        tickvals = [
            0, 1, 2, 3,
            5, 6, 7, 8,
            10, 11, 12, 13,
            15, 16, 17, 18,
            20, 21, 22, 23,
            25, 26, 27, 28,
        ]

        ticktext = (
            variations
            + variations
            + variations
            + variations
            + variations
            + variations
        )

        fig.update_xaxes(

            tickmode="array",

            tickvals=tickvals,

            ticktext=ticktext,

            tickangle=-45,

            range=[
                -0.5,
                28.5
            ],

            row=row,
            col=1,
        )

    # =====================================================
    # Layout
    # =====================================================

    fig.update_layout(

        title=(
            f"{metric.title()} Comparison "
            "by Class, Model and Variation"
        ),

        barmode="overlay",

        template="plotly_white",

        height=280 * n_classes,

        legend=dict(
            title="Variation",

            orientation="h",

            yanchor="bottom",
            y=1.02,

            xanchor="left",
            x=0,
        ),

        margin=dict(
            l=80,
            r=40,
            t=140,
            b=100,
        ),
    )

    return fig


In [135]:
fig = create_class_subplots(
    all_combined,
    metric="f1-score"
)
fig.write_html(
        r"C:\Users\alrazz\Downloads\Results\Modified Files\Perclass results\F1_per_class.html",
        include_plotlyjs=True
    )



fig.show()
